# Hylogic tutorial

Propositional and first-order logic in [Hy](https://hylang.org). This notebook runs on the ordinary Python kernel: the `%%hy` cell magic runs a cell as Hy code, and the value of its last form is the cell's output.

In [1]:
%load_ext hylogic.ipython

## Truth tables

Formulas are quoted Hy code. Logical symbols and ASCII names both work: `→` or `=>`, `∧` or `and`, and so on. A truth table can show several formulas side by side.

In [2]:
%%hy
(import hylogic [truth-table])
(truth-table '(→ (∧ P (→ P Q)) Q) '(⊕ P Q))

P,Q,(P ∧ (P → Q)) → Q,P ⊕ Q
⊤,⊤,⊤,⊥
⊤,⊥,⊤,⊤
⊥,⊤,⊤,⊤
⊥,⊥,⊤,⊥


The first formula is true in every row, so it is a tautology. Truth tables also answer whether two formulas say the same thing.

In [3]:
%%hy
(import hylogic.truth [tautology? equivalent?])
[(tautology? '(=> (=> (=> p q) p) p))          ; Peirce's law
 (equivalent? '(¬ (∧ P Q)) '(∨ (¬ P) (¬ Q)))]   ; De Morgan

[True, True]

## Arguments

An argument is valid when the conclusion is true whenever all premises are. `defargument` takes the formulas as written, and the meanings of the letters are shown in the explanation.

In [4]:
%%hy
(require hylogic.argument [defargument])
(defargument modus-tollens
  :premises [(→ P Q) (¬ Q)]
  :conclusion (¬ P)
  :meanings {P "it rains" Q "the street is wet"})
(print (.explain modus-tollens))

  P → Q
  ¬Q
  ───────
∴ ¬P

  P: it rains
  Q: the street is wet

Valid, by truth table.


An invalid argument gets a counterexample: an assignment where every premise is true and the conclusion is false.

In [5]:
%%hy
(import hylogic [Argument])
(print (.explain (Argument ['(→ P Q) 'Q] 'P)))

  P → Q
  Q
  ───────
∴ P

Invalid, by truth table.
Counterexample: P = ⊥, Q = ⊤


## Exercises

The exercise set has sentences to formalize and arguments to judge. A wrong answer gets a situation where it and the sentence come apart.

In [6]:
%%hy
(import hylogic.exercise-sets [BY-ID check-answer])
(print (get BY-ID "F2" "text"))
(print (check-answer "F2" '(→ P Q)))
(print (check-answer "F2" '(→ Q P)))

The street is wet only if it rains.
Not quite. Suppose “it rains” is false and “the street is wet” is true. Then your formula is true, but the sentence is false.
Correct. Q → P says the same as the sentence.


For first-order sentences the explanation is a small world.

In [7]:
%%hy
(print (get BY-ID "V9" "text"))
(print (check-answer "V9" True))

Everybody loves somebody. So there is somebody whom everybody loves.
Not quite. The argument is invalid. In this world the premises are true and the conclusion is false:
Domain: d1, d2
  Loves holds for: (d1 d1), (d2 d2)


## Models

`evaluate` finds the truth value of a formula in a model, with quantifiers ranging over a finite domain.

In [8]:
%%hy
(import hylogic [evaluate])
(setv model {"Man" #{"socrates" "plato"} "Mortal" #{"socrates" "plato" "fido"}})
[(evaluate '(∀ [x] (→ (Man x) (Mortal x))) model ["socrates" "plato" "fido"])
 (evaluate '(∀ [x] (→ (Mortal x) (Man x))) model ["socrates" "plato" "fido"])]

[True, False]

## Proofs

For first-order logic, `prove` searches for a contradiction between the premises and the negated goal by resolution, and prints the proof.

In [9]:
%%hy
(import hylogic [prove])
(setv result (prove '(Kills curiosity tuna)
                    ['(∃ [x] (∧ (Dog x) (Owns jack x)))
                     '(∀ [x] (→ (∃ [y] (∧ (Dog y) (Owns x y))) (AnimalLover x)))
                     '(∀ [x] (→ (AnimalLover x) (∀ [y] (→ (Animal y) (¬ (Kills x y))))))
                     '(∨ (Kills jack tuna) (Kills curiosity tuna))
                     '(Cat tuna)
                     '(∀ [x] (→ (Cat x) (Animal x)))]))
(for [line (.proof result)] (print line))

   1. (Dog sk8)                                    axiom
   2. (Owns jack sk8)                              axiom
   3. (AnimalLover ?x9) | (not (Dog ?y10)) | (not (Owns ?x9 ?y10)) axiom
   4. (not (Animal ?y12)) | (not (AnimalLover ?x11)) | (not (Kills ?x11 ?y12)) axiom
   5. (Kills curiosity tuna) | (Kills jack tuna)   axiom
   6. (Cat tuna)                                   axiom
   7. (Animal ?x13) | (not (Cat ?x13))             axiom
   8. (not (Kills curiosity tuna))                 negated goal
   9. (Animal tuna)                                resolve 7, 6
  10. (Kills jack tuna)                            resolve 5, 8
  12. (AnimalLover jack) | (not (Dog sk8))         resolve 3, 2
  13. (AnimalLover jack)                           resolve 12, 1
  16. (not (Animal tuna)) | (not (AnimalLover jack)) resolve 4, 10
  21. (not (AnimalLover jack))                     resolve 16, 9
  26. □                                            resolve 21, 13


## Larger problems

Truth tables double with every letter. Above ten letters, arguments are checked with a SAT solver, and `to-smtlib` writes the same question for solvers such as Z3.

In [10]:
%%hy
(import hylogic [to-smtlib])
(print (to-smtlib ['(∀ [x] (→ (Man x) (Mortal x))) '(Man socrates)] '(Mortal socrates)))

(set-logic ALL)
(declare-sort U 0)
(declare-const |socrates| U)
(declare-fun |Man| (U) Bool)
(declare-fun |Mortal| (U) Bool)
(assert (forall ((|x| U)) (=> (|Man| |x|) (|Mortal| |x|))))
(assert (|Man| |socrates|))
(assert (not (|Mortal| |socrates|)))
(check-sat)


## All exercises

In [11]:
%%hy
(import hylogic.exercise-sets [EXERCISES])
(for [e EXERCISES] (print (get e "id") (get e "text")))

F1 If it rains, the street is wet.
F2 The street is wet only if it rains.
F3 It rains unless the sun shines.
F4 Neither the bus nor the train is late.
F5 The bus and the train are not both late.
F6 The alarm sounds if and only if the door opens.
F7 If the door opens and the code is wrong, the alarm sounds.
F8 The meeting is on Monday or on Tuesday, but not on both.
F9 You get a refund only if you have the receipt and return the item within 30 days.
F10 The program stops whenever its input is finite.
F11 Every student reads some book.
F12 Some book is read by every student.
F13 No cat likes water.
F14 Only members can vote.
F15 Everybody loves somebody.
F16 There is somebody whom everybody loves.
V1 If it rains, the street is wet. It rains. So the street is wet.
V2 If it rains, the street is wet. The street is wet. So it rains.
V3 If it rains, the street is wet. It does not rain. So the street is not wet.
V4 If it rains, the street is wet. The street is not wet. So it does not rain.
V5 